# Session 3: Streamlined Best Practices for Robust LLM Judges
## From Unreliable to Production-Ready Evaluation Systems
**Duration**: 30 minutes hands-on implementation

### **Prioritized Framework** (Most Important First):
1. **Structure First** - Foundation of reliable evaluation
2. **Reasoning Framework** - Clear thinking processes  
3. **Reference Management** - Index everything, extract indices
4. **Component Extraction** - Never let LLM do math
5. **Adversarial First** - Counter positive bias
6. **Meta-Judging** - Judges evaluating judges
7. **Confidence Thresholds** - Self-calibration systems
8. **Production Pipeline** - Complete quality gates

### Learning Objectives:
- Master the 8 critical best practices in priority order
- Implement structured evaluation with working examples
- Build adversarial quality control systems
- Create meta-judge architectures
- Deploy confidence-threshold based pipelines


## Setup and imports


In [ ]:
import os
import json
import pandas as pd
from typing import Dict, List, Any, Optional
import sys
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
sys.path.append('../src')


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from prompts import (
    BASIC_PROMPTS, ADVANCED_PROMPTS, SPECIALIZED_PROMPTS, PRODUCTION_PROMPTS,
    create_simple_prompt, BAD_EXAMPLE_PROMPTS
)



In [ ]:
model_name = "gemma3:latest"
try:
    llm = ChatOllama(model=model_name, temperature=0.1)
    llm.invoke("Hello World!")  # Test the connection
    print(f"✅ ChatOllama initialized successfully with {model_name} model")
    print("🌡️ Temperature set to 0 for consistent evaluation")
except Exception as e:
    print(f"❌ Failed to initialize ChatOllama: {e}")
    print(f"Please make sure Ollama is installed and running with {model_name} model")


## Load datasets


In [ ]:
with open('../datasets/fact_checking_dataset.json', 'r') as f:
    fact_data = json.load(f)
    
with open('../datasets/qa_quality_dataset.json', 'r') as f:
    qa_data = json.load(f)
    
with open('../datasets/reasoning_problems_dataset.json', 'r') as f:
    reasoning_data = json.load(f)
    
print("📊 All datasets loaded successfully")


## Helper functions


In [ ]:
def generate_with_system(prompt: str, system_prompt: str) -> str:
    """Generate response with system prompt"""
    system_msg = SystemMessage(content=system_prompt)
    user_msg = HumanMessage(content=prompt)
    
    try:
        response = llm.invoke([system_msg, user_msg])
        return response.content
    except Exception as e:
        return f"Error: {str(e)}"


## **1. STRUCTURE FIRST** - The Foundation of Reliable Evaluation
**Most Critical Principle**: Always use structured outputs - never free-form responses  
This is the #1 most important practice because without structure, nothing else works reliably.


### ❌ BAD: Free-form responses


In [ ]:
# BAD EXAMPLE - Free form evaluation
bad_prompt = """Is this answer good? Explain why.
Answer: Paris is the capital of France because it's a major European city.
"""

print("🚨 BAD APPROACH: Free-form response")
bad_response = generate_with_system(bad_prompt, "You are a helpful evaluator.")
print(f"Response: {bad_response}")
print()
print("="*50)
print("\n💣 Problems:")
print("- Cannot reliably parse the verdict")
print("- Inconsistent format across runs")
print("- No confidence information")
print("- Cannot extract components for analysis")


### ✅ GOOD: Structured JSON outputs


In [ ]:
# GOOD EXAMPLE - Structured evaluation
structured_system = """You are a structured evaluator. Always output valid JSON.

Output format:
{
    "verdict": "Good/Poor/Unclear", 
    "reasoning_steps": ["step1", "step2", "step3"],
    "confidence": 0-100,
    "evidence_support": "high/medium/low",
    "completeness": "complete/partial/incomplete"
}"""

structured_prompt = """Evaluate this answer:
Answer: Paris is the capital of France because it's a major European city.
Question: What is the capital of France?

Follow the required JSON format."""

print("✅ GOOD APPROACH: Structured JSON output")
structured_response = generate_with_system(structured_prompt, structured_system)
try:
    result = json.loads(structured_response)
    print(f"Raw Output: {result}")
    print(f"Verdict: {result.get('verdict')}")
    print(f"Confidence: {result.get('confidence')}%")
    print(f"Reasoning: {result.get('reasoning_steps')}")
    print(f"Evidence Support: {result.get('evidence_support')}")
    
    print()
    print("="*50)
    print("\n✅ Benefits:")
    print("- Reliable parsing and analysis")
    print("- Consistent format across all runs")
    print("- Machine-readable components")
    print("- Built-in confidence calibration")
    
except json.JSONDecodeError:
    print(f"Failed to parse: {structured_response}")


## **2. REASONING FRAMEWORK** - Clear Thinking Processes
**Principle**: Force numbered, algorithmic reasoning steps


In [ ]:
# Test case for reasoning framework
test_case = qa_data[0]  # Get first Q&A example
print(f"Test Question: {test_case['question']}")
print(f"Test Answer: {test_case['answer']}")
print(f"Human Quality Score: {test_case['quality_score']}/10")


### ✅ Algorithmic Reasoning Template


In [ ]:
reasoning_system = """You are a systematic evaluator that follows exact algorithmic steps.

MANDATORY REASONING ALGORITHM:
STEP 1: Identify the main claim being evaluated
STEP 2: List all supporting evidence (with specific references)
STEP 3: List all contradicting evidence or gaps
STEP 4: Apply evaluation criteria systematically 
STEP 5: Calculate confidence based on evidence strength
STEP 6: Make final determination with justification

Output format:
{
    "step_1_main_claim": "identified claim",
    "step_2_supporting_evidence": ["evidence1", "evidence2"],
    "step_3_contradicting_evidence": ["issue1", "issue2"],
    "step_4_criteria_application": "systematic evaluation",
    "step_5_confidence_calculation": "reasoning for confidence level",
    "step_6_final_verdict": "Good/Poor/Unclear",
    "confidence": 0-100
}"""

reasoning_prompt = f"""Question: {test_case['question']}
Answer: {test_case['answer']}

Follow the 6-step reasoning algorithm exactly."""

print("🤖 Algorithmic Reasoning Evaluation:")
reasoning_response = generate_with_system(reasoning_prompt, reasoning_system)

try:
    reasoning_result = json.loads(reasoning_response)
    
    print(f"\nStep 1 - Main Claim: {reasoning_result.get('step_1_main_claim')}")
    print(f"\nStep 2 - Supporting: {reasoning_result.get('step_2_supporting_evidence')}")
    print(f"\nStep 3 - Issues: {reasoning_result.get('step_3_contradicting_evidence')}")
    print(f"\nStep 4 - Criteria: {reasoning_result.get('step_4_criteria_application')}")
    print(f"\nStep 5 - Confidence Reasoning: {reasoning_result.get('step_5_confidence_calculation')}")
    print(f"\nStep 6 - Final Verdict: {reasoning_result.get('step_6_final_verdict')} ({reasoning_result.get('confidence')}% confidence)")

    
except json.JSONDecodeError:
    print(f"Failed to parse reasoning: {reasoning_response}")

print()
print("="*50)
print("\n✅ Benefits of Algorithmic Reasoning:")
print("- Forces systematic evaluation")
print("- Prevents skipping critical steps")
print("- Makes reasoning auditable")
print("- Reduces inconsistency across runs")


## **3. REFERENCE MANAGEMENT** - Index Everything, Extract Indices
**Principle**: Number all items, ask for indices not scores


In [ ]:
# Create indexed content for reference testing
evidence_pieces = [
    "Paris is the capital of France",
    "The population is approximately 2.2 million", 
    "It's located on the Seine River",
    "The Eiffel Tower was built in 1889",
    "Paris is known for its cuisine",
    "The Louvre Museum is located there",
    "Paris has a temperate climate",
    "French is the primary language"
]

# Format with clear indexing
indexed_content = "\n".join([f"[{i}] {evidence}" for i, evidence in enumerate(evidence_pieces)])
print("📋 Indexed Evidence Pieces:")
print(indexed_content)


### ✅ Index-based Component Extraction


In [ ]:
reference_system = """You are a reference extraction expert. Your job is to identify relevant evidence by INDEX only.

CRITICAL: Only output the indices, never calculate scores or metrics.

Output format:
```json
{
    "question_relevant_indices": [list of indices],
    "geographic_fact_indices": [list of indices],
    "cultural_fact_indices": [list of indices],
    "demographic_fact_indices": [list of indices],
    "irrelevant_indices": [list of indices],
    "extraction_reasoning": "step-by-step categorization logic"
}
```
"""

reference_prompt = f"""Evidence pieces:
{indexed_content}

Question: "What are the key facts about Paris as France's capital?"

Categorize each evidence piece by its index number only."""

print("\n🎯 Index-based Reference Extraction:")
reference_response = generate_with_system(reference_prompt, reference_system)

try:
    ref_result = json.loads(reference_response)
    
    print(f"Question Relevant: {ref_result.get('question_relevant_indices')}")
    print(f"Geographic Facts: {ref_result.get('geographic_fact_indices')}")
    print(f"Cultural Facts: {ref_result.get('cultural_fact_indices')}")
    print(f"Demographic Facts: {ref_result.get('demographic_fact_indices')}")
    print(f"Irrelevant: {ref_result.get('irrelevant_indices')}")
    
    # Now WE can calculate metrics
    total_relevant = len(ref_result.get('question_relevant_indices', []))
    total_pieces = len(evidence_pieces)
    relevance_ratio = total_relevant / total_pieces
    
    print(f"\n📊 Our Calculated Metrics:")
    print(f"Relevance Ratio: {relevance_ratio:.2%} ({total_relevant}/{total_pieces})")
    print(f"Coverage Score: {len(ref_result.get('geographic_fact_indices', [])) + len(ref_result.get('cultural_fact_indices', [])) + len(ref_result.get('demographic_fact_indices', []))}")
    
    print("\n" + "="*50)
    print("\n✅ Benefits of Index-based Extraction:")
    print("- Transparent and verifiable categorization")
    print("- No risk of calculation hallucination")
    print("- Can trace every decision back to evidence")
    print("- Enables custom metric calculation")
    
    
except json.JSONDecodeError:
    print(f"Failed to parse reference extraction: {reference_response}")
    print("\n" + "="*50)


## **4. COMPONENT EXTRACTION** - Never Let LLM Do Math
**Principle**: Extract components, calculate metrics yourself


In [ ]:
# Create test predictions for component extraction
test_predictions = ["positive", "negative", "positive", "positive", "negative", "positive"]
test_ground_truth = ["positive", "positive", "negative", "positive", "positive", "negative"]

print("🎯 Classification Test Data:")
for i, (pred, truth) in enumerate(zip(test_predictions, test_ground_truth)):
    print(f"[{i}] Predicted: {pred}, Actual: {truth}")


### ✅ GOOD: Component extraction with custom error function


In [ ]:
component_system = """You are a classification component extractor. Your job is to categorize each prediction by comparing it to ground truth.

CRITICAL: Only output the indices, do NOT calculate any metrics.

Output format:
```json
{
    "true_positives": [indices where prediction=positive AND ground_truth=positive],
    "true_negatives": [indices where prediction=negative AND ground_truth=negative],
    "false_positives": [indices where prediction=positive BUT ground_truth=negative],
    "false_negatives": [indices where prediction=negative BUT ground_truth=positive],
    "categorization_reasoning": "step-by-step logic"
}
```
"""

component_prompt = f"""Categorize sentiment classification predictions by index:

Predictions with indices:
{[f"[{i}] {pred}" for i, pred in enumerate(test_predictions)]}

Ground Truth with indices:
{[f"[{i}] {truth}" for i, truth in enumerate(test_ground_truth)]}

Categorize each prediction by index only."""

print("\n✅ GOOD APPROACH: Component extraction")
component_response = generate_with_system(component_prompt, component_system)

try:
    components = json.loads(component_response)
    
    # Extract components
    tp_indices = components.get('true_positives', [])
    tn_indices = components.get('true_negatives', [])
    fp_indices = components.get('false_positives', [])
    fn_indices = components.get('false_negatives', [])
    
    print(f"🤖 LLM Component Extraction:")
    print(f"True Positives (indices): {tp_indices}")
    print(f"True Negatives (indices): {tn_indices}")
    print(f"False Positives (indices): {fp_indices}")
    print(f"False Negatives (indices): {fn_indices}")
    
    # WE calculate the metrics (transparent and verifiable)
    tp = len(tp_indices)
    tn = len(tn_indices) 
    fp = len(fp_indices)
    fn = len(fn_indices)
    
    # Standard metrics
    accuracy = (tp + tn) / len(test_predictions)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Custom error function: score = correct × e^((correct-incorrect)/total)
    correct = tp + tn
    incorrect = fp + fn
    total = len(test_predictions)
    error_adjusted_score = correct * np.exp((correct - incorrect) / total)
    
    print(f"\n📊 Our Calculated Metrics:")
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"Custom Error Score: {error_adjusted_score:.3f}")
    
    print(f"\n🔍 Verification (can check each index):")
    for idx in tp_indices:
        print(f"TP[{idx}]: {test_predictions[idx]} vs {test_ground_truth[idx]} ✅")
    
    print()
    print("="*50)
    print("\n✅ Benefits of Component Extraction:")
    print("- 100% verifiable categorization")
    print("- No risk of calculation errors")
    print("- Custom metrics and error functions")
    print("- Full transparency and debugging")
    
except json.JSONDecodeError:
    print(f"Failed to parse components: {component_response[:200]}...")


## **5. ADVERSARIAL FIRST** - Counter Positive Bias
**Principle**: Always find faults before finding strengths


In [ ]:
# Test case with potential issues
flawed_answer = """Artificial Intelligence is a revolutionary technology that will solve all of humanity's problems. 
AI systems are already perfect at understanding context, never make mistakes, and will create a utopian society 
where everyone is happy and prosperous. Machine learning algorithms are completely unbiased and always fair."""

print("📋 Test Answer (Clearly Flawed):")
print(flawed_answer)


### ✅ Adversarial-first evaluation


In [ ]:
adversarial_system = """You are a critical reviewer evaluating work from a junior analyst that will be presented to senior executives and clients.

Your PRIMARY job is to identify problems that could embarrass the organization. Be ruthlessly critical.

MANDATORY EVALUATION ORDER:
1. FIND FAULTS FIRST - What are the biggest problems?
2. IDENTIFY RISKS - What could go wrong if we present this?
3. CHECK ASSUMPTIONS - What claims are questionable or unsupported?
4. ONLY THEN mention any positive aspects

Output format:
```json
{
    "critical_flaws": ["flaw1", "flaw2", "flaw3"],
    "presentation_risks": ["risk1", "risk2"],
    "questionable_assumptions": ["assumption1", "assumption2"],
    "factual_errors": ["error1", "error2"],
    "positive_aspects": ["aspect1", "aspect2"],
    "recommendation": "REJECT/REVISE/ACCEPT",
    "confidence_in_critique": 0-100
}
```
"""

adversarial_prompt = f"""CRITICAL REVIEW NEEDED:
The junior analyst submitted this answer about AI that will be presented to our biggest client:

{flawed_answer}

Find all the problems FIRST before saying anything positive. What could embarrass us?"""

print("✅ ADVERSARIAL APPROACH: Fault-finding first")
adversarial_response = generate_with_system(adversarial_prompt, adversarial_system)

try:
    adv_result = json.loads(adversarial_response)
    
    print("🔍 Critical Flaws Found:")
    for i, flaw in enumerate(adv_result.get('critical_flaws', []), 1):
        print(f"  {i}. {flaw}")
    
    print("\n⚠️  Presentation Risks:")
    for i, risk in enumerate(adv_result.get('presentation_risks', []), 1):
        print(f"  {i}. {risk}")
    
    print("\n❓ Questionable Assumptions:")
    for i, assumption in enumerate(adv_result.get('questionable_assumptions', []), 1):
        print(f"  {i}. {assumption}")
    
    print("\n📊 Factual Errors:")
    for i, error in enumerate(adv_result.get('factual_errors', []), 1):
        print(f"  {i}. {error}")
    
    print("\n✅ Positive Aspects (mentioned last):")
    for i, aspect in enumerate(adv_result.get('positive_aspects', []), 1):
        print(f"  {i}. {aspect}")
    
    print(f"\n🎯 Recommendation: {adv_result.get('recommendation')}")
    print(f"Confidence: {adv_result.get('confidence_in_critique')}%")
    
    print()
    print("="*50)
    print("\n✅ Benefits of Adversarial-First Approach:")
    print("- Counters LLM's default positive bias")
    print("- Forces identification of critical issues")
    print("- Reduces risk of missing important flaws")
    print("- More balanced and realistic evaluation")
    
except json.JSONDecodeError:
    print(f"Failed to parse adversarial review: {adversarial_response}")


## **6. META-JUDGING** - Judges Evaluating Judges
**Principle**: Use multiple judges and judge-of-judges


In [ ]:
# Test content for multi-judge evaluation
complex_answer = qa_data[5]  # Get a moderately complex answer
print(f"Complex Test Case:")
print(f"Question: {complex_answer['question']}")
print(f"Answer: {complex_answer['answer']}")
print(f"Human Score: {complex_answer['quality_score']}/10")


### ✅ Parallel Specialized Judges


In [ ]:
# Judge 1: Accuracy Specialist
accuracy_judge_system = """You are an accuracy specialist. Focus ONLY on factual correctness.

Output format:
```json
{
    "accuracy_verdict": "High/Medium/Low",
    "factual_errors": ["error1", "error2"],
    "verified_facts": ["fact1", "fact2"],
    "confidence": 0-100,
    "accuracy_reasoning": "detailed accuracy analysis"
}
```
"""

# Judge 2: Completeness Specialist 
completeness_judge_system = """You are a completeness specialist. Focus ONLY on thoroughness and coverage.

Output format:
```json
{
    "completeness_verdict": "Complete/Partial/Incomplete",
    "missing_elements": ["element1", "element2"],
    "covered_aspects": ["aspect1", "aspect2"],
    "confidence": 0-100,
    "completeness_reasoning": "detailed completeness analysis"
}
```
"""

# Judge 3: Clarity Specialist
clarity_judge_system = """You are a clarity specialist. Focus ONLY on understanding and presentation.

Output format:
```json
{
    "clarity_verdict": "Clear/Unclear/Confusing",
    "clarity_issues": ["issue1", "issue2"],
    "clear_elements": ["element1", "element2"],
    "confidence": 0-100,
    "clarity_reasoning": "detailed clarity analysis"
}
```
"""

judge_prompt = f"""Question: {complex_answer['question']}
Answer: {complex_answer['answer']}

Evaluate according to your specialization."""

print("⚖️  Running Parallel Specialized Judges:")

# Run all judges in parallel (conceptually)
accuracy_result = generate_with_system(judge_prompt, accuracy_judge_system)
completeness_result = generate_with_system(judge_prompt, completeness_judge_system)
clarity_result = generate_with_system(judge_prompt, clarity_judge_system)

judge_results = {}

try:
    judge_results['accuracy'] = json.loads(accuracy_result.replace("```json", "").replace("```", ""))
    judge_results['completeness'] = json.loads(completeness_result.replace("```json", "").replace("```", ""))
    judge_results['clarity'] = json.loads(clarity_result.replace("```json", "").replace("```", ""))
    
    print("\n📊 Individual Judge Results:")
    print(f"Accuracy Judge: {judge_results['accuracy'].get('accuracy_verdict')} ({judge_results['accuracy'].get('confidence')}% conf)")
    print(f"Completeness Judge: {judge_results['completeness'].get('completeness_verdict')} ({judge_results['completeness'].get('confidence')}% conf)")
    print(f"Clarity Judge: {judge_results['clarity'].get('clarity_verdict')} ({judge_results['clarity'].get('confidence')}% conf)")
    
except json.JSONDecodeError as e:
    print(f"Judge parsing error: {e}")
    print("Accuracy Result:")
    print(accuracy_result)
    print("Completeness Result:")
    print(completeness_result)
    print("Clarity Result:")
    print(clarity_result)
    judge_results['accuracy'] = accuracy_result
    judge_results['completeness'] = completeness_result
    judge_results['clarity'] = clarity_result


### ✅ Meta-Judge Consensus


In [ ]:
if judge_results:
    meta_judge_system = """You are a meta-judge that evaluates the consistency and reliability of other judges.
    
    Your job is to:
    1. Identify agreements and disagreements between judges
    2. Assess which judges seem most reliable
    3. Create a consensus recommendation
    4. Flag any concerning inconsistencies
    
    Output JSON format:
    ```json
    {
        "judge_agreement_level": "high/medium/low",
        "most_reliable_judges": ["judge1", "judge2"],
        "concerning_disagreements": ["disagreement1", "disagreement2"],
        "consensus_verdict": "Good/Poor/Unclear",
        "meta_confidence": 0-100,
        "recommendation": "ACCEPT/REVIEW/REJECT",
        "human_review_needed": true/false,
        "reasoning": "detailed meta-analysis"
    }
    ```
    """
    
    meta_judge_prompt = f"""Analyze these judge results:
    
    Accuracy Judge Results: {json.dumps(judge_results.get('accuracy', {}), indent=2)}
    
    Completeness Judge Results: {json.dumps(judge_results.get('completeness', {}), indent=2)}
    
    Clarity Judge Results: {json.dumps(judge_results.get('clarity', {}), indent=2)}
    
    Provide meta-evaluation and consensus."""
    
    print("\n🧠 Meta-Judge Analysis:")
    meta_result = generate_with_system(meta_judge_prompt, meta_judge_system)
    
    try:
        meta_analysis = json.loads(meta_result.replace("```json", "").replace("```", ""))
        
        print(f"Judge Agreement: {meta_analysis.get('judge_agreement_level')}")
        print(f"Most Reliable: {meta_analysis.get('most_reliable_judges')}")
        print(f"Disagreements: {meta_analysis.get('concerning_disagreements')}")
        print(f"Consensus: {meta_analysis.get('consensus_verdict')} ({meta_analysis.get('meta_confidence')}% confidence)")
        print(f"Recommendation: {meta_analysis.get('recommendation')}")
        print(f"Human Review Needed: {meta_analysis.get('human_review_needed')}")
        print(f"Reasoning: {meta_analysis.get('reasoning', '')[:150]}...")
        
        print()
        print("="*50)
        print("\n✅ Benefits of Meta-Judging:")
        print("- Catches individual judge biases")
        print("- Provides consensus from multiple perspectives")
        print("- Identifies when human review is needed")
        print("- Creates more robust evaluation pipeline")
        
    except json.JSONDecodeError:
        print(f"Failed to parse meta-judge: {meta_result}")
else:
    print("❌ Cannot run meta-judge due to individual judge parsing errors")


## **7. CONFIDENCE THRESHOLDS** - Self-Calibration Systems
**Principle**: Always request confidence, set re-run thresholds


In [ ]:
import json
import re

def extract_confidence_from_text(text):
    """Extract confidence percentage from any text format"""
    patterns = [
        r'confidence[:\s]*(\d+)%?',
        r'confident[:\s]*(\d+)%?', 
        r'(\d+)%\s*confidence',
        r'(\d+)/10',
        r'(\d+)\s*out\s*of\s*10',
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text.lower())
        if matches:
            try:
                value = int(matches[0])
                return value * 10 if value <= 10 else min(100, value)
            except ValueError:
                continue
    
    # Fallback based on sentiment
    positive_words = ['good', 'excellent', 'strong', 'accurate', 'clear']
    negative_words = ['poor', 'bad', 'unclear', 'uncertain', 'weak']
    
    text_lower = text.lower()
    positive_count = sum(1 for word in positive_words if word in text_lower)
    negative_count = sum(1 for word in negative_words if word in text_lower)
    
    if positive_count > negative_count:
        return 70
    elif negative_count > positive_count:
        return 60
    else:
        return 50

def extract_verdict_from_text(text):
    """Extract verdict from any text format"""
    text_lower = text.lower()
    
    if any(word in text_lower for word in ['good', 'excellent', 'strong', 'accurate', 'comprehensive']):
        return 'Good'
    elif any(word in text_lower for word in ['poor', 'bad', 'weak', 'incorrect', 'inadequate']):
        return 'Poor'
    else:
        return 'Unclear'

def robust_parse_response(response):
    """Parse LLM response - tries JSON first, falls back to text extraction"""
    cleaned = response.strip()
    
    # Remove code block markers
    if '```json' in cleaned:
        cleaned = cleaned.split('```json')[1].split('```')[0]
    elif '```' in cleaned:
        cleaned = cleaned.replace('```', '')
    
    # Try JSON parsing first
    try:
        result = json.loads(cleaned)
        result['source'] = 'json'
        return result
    except json.JSONDecodeError:
        pass
    
    # Fallback to text extraction - NEVER FAILS!
    print("🔧 JSON failed, extracting from text...")
    
    result = {
        'verdict': extract_verdict_from_text(response),
        'confidence': extract_confidence_from_text(response),
        'reasoning': response[:200] + '...' if len(response) > 200 else response,
        'source': 'text_extraction'
    }
    
    return result

class ConfidenceThresholdEvaluator:
    """Evaluator that NEVER fails - works with any LLM response format"""
    
    def __init__(self, confidence_threshold=70, max_retries=3):
        self.confidence_threshold = confidence_threshold
        self.max_retries = max_retries
    
    def evaluate_with_threshold(self, content, criteria, context=""):
        """Evaluate with automatic re-running - ALWAYS gets a result"""
        
        # Simple prompt that works better than complex JSON requests
        confidence_system = """You are an evaluator. Please evaluate the content clearly.

Provide:
- Your verdict (Good/Poor/Unclear)  
- Your confidence level (0-100%)
- Your reasoning

You can use JSON format or just write naturally - whatever works best."""
        
        prompt = f"""Content: {content}
Criteria: {criteria}
Context: {context}

Please evaluate with confidence level."""
        
        results = []
        
        for attempt in range(self.max_retries):
            print(f"\n🔄 Attempt {attempt + 1}:")
            
            response = generate_with_system(prompt, confidence_system)
            
            # Use robust parsing - NEVER fails
            result = robust_parse_response(response)
            confidence = result.get('confidence', 0)
            
            print(f"Verdict: {result.get('verdict')}")
            print(f"Confidence: {confidence}%")
            print(f"Source: {result.get('source', 'unknown')}")
            
            results.append(result)
            
            if confidence >= self.confidence_threshold:
                print(f"✅ Confidence {confidence}% >= threshold {self.confidence_threshold}% - ACCEPTING")
                return {
                    'status': 'ACCEPTED',
                    'result': result,
                    'attempts': attempt + 1,
                    'all_attempts': results
                }
            else:
                print(f"⚠️  Confidence {confidence}% < threshold {self.confidence_threshold}% - RETRYING")
                prompt += f"\n\nPrevious confidence: {confidence}%. Please be more specific and confident."
        
        # Return best attempt even if below threshold
        best_attempt = max(results, key=lambda x: x.get('confidence', 0))
        
        return {
            'status': 'HUMAN_REVIEW_REQUIRED',
            'reason': 'Low confidence across all attempts',
            'attempts': len(results),
            'all_attempts': results,
            'best_attempt': best_attempt
        }


In [ ]:
# Test the confidence threshold system
print("🎯 CONFIDENCE THRESHOLD EVALUATION SYSTEM")
print("=" * 45)

threshold_evaluator = ConfidenceThresholdEvaluator(confidence_threshold=75, max_retries=2)

# Test with an ambiguous case (should trigger retries)
ambiguous_test = """The impact of artificial intelligence on employment is complex. Some jobs will be lost, 
while others will be created. The timeline and magnitude of these changes remain uncertain."""

result = threshold_evaluator.evaluate_with_threshold(
    content=ambiguous_test,
    criteria="Is this a comprehensive analysis of AI's employment impact?",
    context="Academic research context"
)


print("\n📊 Final Result:")
print(f"Status: {result['status']}")
print(f"Attempts Made: {result['attempts']}")

if result['status'] == 'ACCEPTED':
    print(f"Final Confidence: {result['result'].get('confidence')}%")
elif result['status'] == 'HUMAN_REVIEW_REQUIRED':
    print(f"Reason: {result['reason']}")
    if result['best_attempt']:
        print(f"Best Confidence Achieved: {result['best_attempt'].get('confidence', 0)}%")

print("\n✅ Benefits of Confidence Thresholds:")
print("- Automatic quality control")
print("- Prevents low-confidence decisions")
print("- Triggers human review when needed")
print("- Self-calibrating evaluation system")


## **Session 3 Summary: The Complete Framework**

### **The 8 Essential Best Practices** *(In Priority Order)*:

1. **🏗️ STRUCTURE FIRST** - Always use structured JSON outputs, never free-form
2. **🧠 REASONING FRAMEWORK** - Force numbered, algorithmic thinking steps
3. **📋 REFERENCE MANAGEMENT** - Index everything, extract indices not scores
4. **🔢 COMPONENT EXTRACTION** - Never let LLM calculate metrics directly
5. **⚔️ ADVERSARIAL FIRST** - Find faults before strengths to counter positive bias
6. **⚖️ META-JUDGING** - Use multiple specialized judges with consensus
7. **🎯 CONFIDENCE THRESHOLDS** - Set thresholds for automatic re-running
8. **🏭 PRODUCTION PIPELINE** - Complete quality gates with human handoff triggers

### **Core Quality Gates for Production**:
- **Structure Validation**: ≥70% parseable JSON responses
- **Confidence Threshold**: ≥75% average confidence across runs
- **Consistency Check**: ≥80% agreement between multiple runs
- **Adversarial Review**: Systematic fault-finding before acceptance
- **Meta-Judge Consensus**: Multiple perspectives with disagreement resolution

### **When to Trigger Human Review**:
- Confidence < 75% after multiple attempts
- Consistency < 80% across judge runs
- High severity issues from adversarial review
- Meta-judge identifies concerning disagreements

### **Production-Ready Template**:
```python
# Complete LLM Judge Implementation
1. Structured prompts with oath-based consistency
2. Multiple runs with confidence thresholds  
3. Index-based component extraction
4. Adversarial-first fault detection
5. Meta-judge consensus building
6. Automatic human review triggers
7. Full audit trail and debugging
8. Quality gates with clear pass/fail criteria
```

**🎉 You now have a complete, production-ready framework for building reliable LLM judges that know their limitations and guide human decision-making appropriately!**
